# 02 — Entraînement du modèle Seq2Seq

Ce notebook est un **orchestrateur** :
- La logique d'entraînement vit dans `src/train.py`
- Les hyperparamètres viennent de `configs/base.yaml`
- Ce notebook instancie, appelle, sauvegarde

## Chargement de la config

In [4]:
import yaml

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)

print(cfg)

{'data': {'n_train': 50000, 'min_freq': 2, 'max_len': 50, 'batch_size': 64}, 'encoder': {'embed_dim': 256, 'hidden_dim': 512, 'n_layers': 2, 'dropout': 0.3}, 'decoder': {'embed_dim': 256, 'hidden_dim': 512, 'n_layers': 2, 'n_heads': 8, 'dropout': 0.3}, 'optimizer': {'lr': 0.001}, 'loss': {'label_smoothing': 0.1}, 'training': {'n_epochs': 10, 'clip': 1.0, 'checkpoint_dir': '../checkpoints', 'checkpoint_every': 1}}


## Imports

In [5]:
import sys, os
sys.path.append("../src")

import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import spacy
from tqdm import tqdm

from encoder  import Encoder
from decoder  import Decoder
from seq2seq  import Seq2Seq
from data     import Vocab, TranslationDataset, collate_fn
from loss     import Seq2SeqLoss
from train    import train_epoch, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

Device : cpu


## Données

In [ ]:
ds    = load_dataset("Helsinki-NLP/opus-100", "en-fr")
train = ds["train"]

N = cfg["data"]["n_train"]
src_sentences = [ex["translation"]["en"] for ex in train.select(range(N))]
trg_sentences = [ex["translation"]["fr"] for ex in train.select(range(N))]

In [ ]:
nlp_en = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

def batch_tokenize(nlp, sentences, batch_size=512):
    """Tokenise en batch via nlp.pipe() — beaucoup plus rapide que phrase par phrase."""
    return [
        [tok.text.lower() for tok in doc]
        for doc in tqdm(nlp.pipe(sentences, batch_size=batch_size), total=len(sentences))
    ]

src_tokens = batch_tokenize(nlp_en, src_sentences)
trg_tokens = batch_tokenize(nlp_fr, trg_sentences)

In [ ]:
src_vocab = Vocab.build(src_tokens, min_freq=2)
trg_vocab = Vocab.build(trg_tokens, min_freq=2)

print(f"Vocab EN : {len(src_vocab):,} | Vocab FR : {len(trg_vocab):,}")

In [ ]:
dataset      = TranslationDataset(src_sentences, trg_sentences, src_vocab, trg_vocab)
train_loader = DataLoader(
    dataset,
    batch_size=cfg["data"]["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
)
print(f"Batches par epoch : {len(train_loader)}")

## Modèle

In [ ]:
encoder = Encoder(vocab_size=len(src_vocab), **cfg["encoder"])
decoder = Decoder(vocab_size=len(trg_vocab), **cfg["decoder"])
model   = Seq2Seq(encoder, decoder, device).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres entraînables : {n_params:,}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), **cfg["optimizer"])
criterion = Seq2SeqLoss(**cfg["loss"])

## Boucle d'entraînement

`train_epoch` et `evaluate` viennent de `src/train.py` — on les appelle directement.

In [ ]:
os.makedirs(cfg["training"]["checkpoint_dir"], exist_ok=True)

def save_checkpoint(epoch, model, optimizer, loss, checkpoint_dir):
    path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch:02d}.pt")
    torch.save({
        "epoch"                : epoch,
        "model_state_dict"     : model.state_dict(),
        "optimizer_state_dict" : optimizer.state_dict(),
        "loss"                 : loss,
    }, path)


train_losses = []
every = cfg["training"]["checkpoint_every"]

for epoch in range(1, cfg["training"]["n_epochs"] + 1):

    # train_epoch gère : forward, loss, backward, gradient clipping, optimizer step
    loss = train_epoch(model, train_loader, optimizer, criterion,
                       clip=cfg["training"]["clip"], device=device)
    train_losses.append(loss)
    print(f"Epoch {epoch:02d} | Loss : {loss:.4f}")

    if epoch % every == 0:
        save_checkpoint(epoch, model, optimizer, loss,
                        cfg["training"]["checkpoint_dir"])

## Courbe de loss

In [ ]:
import matplotlib.pyplot as plt

# TODO : tracer train_losses en fonction des epochs
raise NotImplementedError